# Week 4 — LangGraph sidekick (solarinayo)



In [ ]:
import json
import uuid
from datetime import datetime
from pathlib import Path
from typing import Annotated, Any, Dict, List, Optional

import gradio as gr
import nest_asyncio
from dotenv import load_dotenv
from duckduckgo_search import DDGS
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

nest_asyncio.apply()

In [ ]:
import os

load_dotenv(override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the repo root .env")

# Sandbox for file tools (relative to cwd — run notebook with cwd = repo root)
SANDBOX = Path("4_langgraph/community_contributions/solarinayo/sandbox")
SANDBOX.mkdir(parents=True, exist_ok=True)
print("Sandbox:", SANDBOX.resolve())

In [ ]:
def ddg_search(query: str) -> str:
    """DuckDuckGo text search; returns JSON snippets (truncated)."""
    with DDGS() as ddgs:
        hits = list(ddgs.text(query, max_results=5))
    return json.dumps(hits, indent=2)[:8000]


web_tool = Tool(
    name="web_search",
    func=ddg_search,
    description="Search the web for fresh facts. Pass a short, specific query.",
)

_wiki = WikipediaAPIWrapper()
wiki_tool = WikipediaQueryRun(api_wrapper=_wiki)

_file_tk = FileManagementToolkit(root_dir=str(SANDBOX.resolve()))
file_tools = _file_tk.get_tools()

tools = file_tools + [web_tool, wiki_tool]
print(f"Loaded {len(tools)} tools (files + web + wikipedia).")

In [ ]:
class ThreeQuestions(BaseModel):
    q1: str = Field(description="First clarifying question")
    q2: str = Field(description="Second clarifying question")
    q3: str = Field(description="Third clarifying question")


class PlanOut(BaseModel):
    steps: List[str] = Field(description="Ordered steps the worker should follow")


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether success criteria are met")
    user_input_needed: bool = Field(
        description="True if the user must answer a question or unblock the assistant"
    )

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool


def human_turns(messages: List[Any]) -> int:
    return sum(1 for m in messages if isinstance(m, HumanMessage))


def format_conversation(messages: List[Any]) -> str:
    lines: List[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            text = m.content or "[tool calls]"
            lines.append(f"Assistant: {text}")
    return "\n".join(lines)


clarify_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(ThreeQuestions)
planner_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(PlanOut)
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)
evaluator_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EvaluatorOutput)

In [ ]:
def route_start(state: State) -> str:
    n = human_turns(state["messages"])
    if n == 1:
        return "clarify"
    if n == 2:
        return "planner"
    return "worker"


def clarify(state: State) -> Dict[str, Any]:
    last = state["messages"][-1].content
    sys = SystemMessage(
        content="You gather exactly three concise clarifying questions before starting work."
    )
    qs = clarify_llm.invoke([sys, HumanMessage(content=last)])
    body = (
        "Before I start, please answer these clarifying questions:\n\n"
        f"1. {qs.q1}\n"
        f"2. {qs.q2}\n"
        f"3. {qs.q3}\n"
    )
    return {"messages": [AIMessage(content=body)]}


def planner(state: State) -> Dict[str, Any]:
    sys = SystemMessage(
        content=(
            "Given the user's task and their answers to clarifying questions, "
            "produce a short ordered plan (steps) for a worker agent with web + wiki + file tools."
        )
    )
    plan = planner_llm.invoke([sys] + state["messages"])
    text = "**Plan**\n" + "\n".join(f"{i+1}. {s}" for i, s in enumerate(plan.steps))
    return {"messages": [AIMessage(content=text)]}


def worker(state: State) -> Dict[str, Any]:
    when = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    system_message = f"""You are a helpful assistant with tools (files in sandbox, web search, Wikipedia).
Work until success criteria are met or you must ask the user something.
Current date/time: {when}

Success criteria:
{state['success_criteria']}

Follow the latest plan in the conversation. Use tools when you need facts or files.
Finish with a clear final answer when done (no question in that case).
If you need the user, start with "Question:".
"""
    if state.get("feedback_on_work"):
        system_message += f"""
Previous attempt did not meet criteria. Evaluator feedback:
{state['feedback_on_work']}
Address this and continue.
"""
    msgs = state["messages"]
    found = False
    for m in msgs:
        if isinstance(m, SystemMessage):
            m.content = system_message
            found = True
    if not found:
        msgs = [SystemMessage(content=system_message)] + msgs
    response = worker_llm_with_tools.invoke(msgs)
    return {"messages": [response]}


def worker_router(state: State) -> str:
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return "evaluator"


def evaluator(state: State) -> Dict[str, Any]:
    last_response = state["messages"][-1].content
    sys = SystemMessage(
        content="You evaluate whether the assistant met the success criteria. Be fair; if they used tools appropriately and answered, accept."
    )
    user = HumanMessage(
        content=(
            f"Conversation:\n{format_conversation(state['messages'])}\n\n"
            f"Success criteria:\n{state['success_criteria']}\n\n"
            f"Last assistant message:\n{last_response}\n"
        )
    )
    ev = evaluator_llm.invoke([sys, user])
    return {
        "messages": [
            {
                "role": "assistant",
                "content": f"Evaluator: {ev.feedback}",
            }
        ],
        "feedback_on_work": ev.feedback,
        "success_criteria_met": ev.success_criteria_met,
        "user_input_needed": ev.user_input_needed,
    }


def route_eval(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    return "worker"

In [ ]:
memory = MemorySaver()
gb = StateGraph(State)
gb.add_node("clarify", clarify)
gb.add_node("planner", planner)
gb.add_node("worker", worker)
gb.add_node("tools", ToolNode(tools=tools))
gb.add_node("evaluator", evaluator)

gb.add_conditional_edges(
    START,
    route_start,
    {"clarify": "clarify", "planner": "planner", "worker": "worker"},
)
gb.add_edge("clarify", END)
gb.add_edge("planner", "worker")
gb.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
gb.add_edge("tools", "worker")
gb.add_conditional_edges("evaluator", route_eval, {"worker": "worker", "END": END})

graph = gb.compile(checkpointer=memory)
print("Graph compiled (clarify → [END] | planner → worker ↔ tools → evaluator).")

In [ ]:
THREAD_ID = str(uuid.uuid4())


async def chat_turn(message: str, criteria: str, history: list):
    if not message.strip():
        return history
    crit = (criteria or "").strip() or "Give a clear, accurate, actionable answer."
    payload = {
        "messages": [HumanMessage(content=message.strip())],
        "success_criteria": crit,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
    }
    cfg = {"configurable": {"thread_id": THREAD_ID}}
    snap = await graph.aget_state(cfg)
    vals = getattr(snap, "values", None) or {}
    n_before = len(vals.get("messages") or [])
    out = await graph.ainvoke(payload, config=cfg)
    msgs = out["messages"]
    new_slice = msgs[n_before:]
    parts = []
    for m in new_slice:
        if isinstance(m, AIMessage) and m.content:
            parts.append(m.content)
        if isinstance(m, dict) and m.get("content"):
            parts.append(str(m["content"]))
    reply = "\n\n---\n\n".join(parts) if parts else "(no assistant text this step)"
    new_hist = list(history)
    new_hist.append([message.strip(), reply])
    return new_hist


with gr.Blocks(title="Week 4 LangGraph — clarify / plan / sidekick") as demo:
    gr.Markdown(
        "### LangGraph sidekick — message 1 = task (get 3 questions); message 2 = your answers; then worker runs."
    )
    crit = gr.Textbox(label="Success criteria (optional)", lines=2)
    chat = gr.Chatbot(label="Conversation", height=420, type="tuples")
    msg = gr.Textbox(label="Your message", lines=3)
    go = gr.Button("Send", variant="primary")
    # Button uses .click(); .submit() exists on Textbox only
    _inputs = [msg, crit, chat]
    _clear = lambda: gr.update(value="")
    go.click(chat_turn, _inputs, [chat]).then(_clear, outputs=msg)
    msg.submit(chat_turn, _inputs, [chat]).then(_clear, outputs=msg)

demo.launch(inbrowser=True)